<a href="https://colab.research.google.com/github/rania04286-r/cv-license-plate-detection/blob/main/CV_Team_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
import zipfile
project_dir = "/content/drive/MyDrive/CV_Team_Project"
dataset_dir = os.path.join(project_dir, "dataset")
zip_files = [
    f for f in os.listdir(dataset_dir)
    if f.lower().endswith(".zip")
]
print("ZIP files found:", zip_files)

ZIP files found: ['License Plate Recognition.v12i.voc.zip']


⚠️ DO NOT RERUN THIS CELL UNDER UNLESS THE DATASET IS MISSING OR INCOMPLETE.

   This cell extracts the original ZIP into Google Drive and may write thousands of files. The dataset has already been extracted.

In [8]:
zip_path = os.path.join(dataset_dir, zip_files[0])
extract_dir = os.path.join(dataset_dir, "extracted")
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)
print("Extraction complete!")
print("Extracted folders:", os.listdir(extract_dir))

Extraction complete!
Extracted folders: ['README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']


In [10]:

from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET

# Update this path if your Drive folder has a different name
DATASET_ROOT = Path(
    "/content/drive/MyDrive/CV_Team_Project/dataset/extracted"
)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for split in ["train", "valid", "test"]:
    split_dir = DATASET_ROOT / split

    if not split_dir.exists():
        print(f"\n{split}: folder not found")
        continue

    images = [
        p for p in split_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]
    xml_files = list(split_dir.rglob("*.xml"))

    print(f"\n--- {split.upper()} ---")
    print("Images:", len(images))
    print("XML annotations:", len(xml_files))

    class_counts = Counter()
    invalid_xml = 0
    invalid_boxes = 0

    for xml_path in xml_files:
        try:
            root = ET.parse(xml_path).getroot()

            for obj in root.findall("object"):
                class_name = obj.findtext("name", "UNKNOWN")
                class_counts[class_name] += 1

                box = obj.find("bndbox")
                if box is None:
                    invalid_boxes += 1
                    continue

                try:
                    xmin = float(box.findtext("xmin"))
                    ymin = float(box.findtext("ymin"))
                    xmax = float(box.findtext("xmax"))
                    ymax = float(box.findtext("ymax"))

                    if xmax <= xmin or ymax <= ymin:
                        invalid_boxes += 1
                except (TypeError, ValueError):
                    invalid_boxes += 1

        except ET.ParseError:
            invalid_xml += 1

    print("Objects by class:", dict(class_counts))
    print("Malformed XML files:", invalid_xml)
    print("Invalid/missing boxes:", invalid_boxes)


--- TRAIN ---
Images: 7057
XML annotations: 7057
Objects by class: {'License_Plate': 7357}
Malformed XML files: 0
Invalid/missing boxes: 0

--- VALID ---
Images: 2048
XML annotations: 2048
Objects by class: {'License_Plate': 2195}
Malformed XML files: 0
Invalid/missing boxes: 0

--- TEST ---
Images: 1020
XML annotations: 1020
Objects by class: {'License_Plate': 1085}
Malformed XML files: 0
Invalid/missing boxes: 0
